# Wildfire Risk ML Training — India

Trains an XGBoost fire-risk classifier using **real historical data**:
- NASA FIRMS (fire detections, VIIRS)
- Open-Meteo Historical Weather API

No fake/synthetic data — this notebook downloads real records each time it runs.

**Before running:** replace `FIRMS_MAP_KEY` below with your own free key from
https://firms.modaps.eosdis.nasa.gov/api/map_key/ (the one below may hit rate
limits if shared across many users).

In [ ]:
!pip install -q xgboost scikit-learn pandas numpy requests joblib matplotlib

In [ ]:
import requests
import pandas as pd
import numpy as np
from datetime import date, timedelta
import time

FIRMS_MAP_KEY = "53a081cd21cb994a04a578b99205dc56"  # replace with your own if this one expires
INDIA_BBOX = "68,8,97,37"  # west, south, east, north (India)
HISTORY_DAYS = 180  # how far back to pull fire data (~6 months)

## 1. Download real fire detections (NASA FIRMS)

In [ ]:
def fetch_firms_chunk(start_date, days=5, source="VIIRS_SNPP_NRT"):
    """FIRMS area API allows max 5 days per request; loop over date to cover HISTORY_DAYS."""
    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{FIRMS_MAP_KEY}/{source}/{INDIA_BBOX}/{days}/{start_date}"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    from io import StringIO
    text = resp.text.strip()
    if len(text.split("\n")) < 2:
        return pd.DataFrame()
    return pd.read_csv(StringIO(text))

all_fires = []
today = date.today()
d = today - timedelta(days=HISTORY_DAYS)
while d <= today:
    try:
        chunk = fetch_firms_chunk(d.isoformat(), days=5)
        if not chunk.empty:
            all_fires.append(chunk)
    except Exception as e:
        print(f"  skip {d}: {e}")
    d += timedelta(days=5)
    time.sleep(0.3)  # stay well under the 5000/10min rate limit

fires_df = pd.concat(all_fires, ignore_index=True) if all_fires else pd.DataFrame()
print(f"Total fire detections downloaded: {len(fires_df)}")
fires_df.head()

## 2. Build a labeled dataset (positive = fire, negative = no fire)

For each fire detection (positive example), sample a random nearby point/date with
NO fire detected (negative example). Then fetch real historical weather for every
point via Open-Meteo's Historical Weather API (no key needed).

In [ ]:
# Positive examples: real fire detections
positives = fires_df[["latitude", "longitude", "acq_date"]].copy()
positives.columns = ["lat", "lon", "date"]
positives["label"] = 1

# Negative examples: random India points/dates NOT in the fire set (same count as positives)
rng = np.random.default_rng(42)
n_neg = len(positives)
neg_lats = rng.uniform(8, 37, n_neg)
neg_lons = rng.uniform(68, 97, n_neg)
neg_dates = [
    (today - timedelta(days=int(x))).isoformat()
    for x in rng.integers(0, HISTORY_DAYS, n_neg)
]
negatives = pd.DataFrame({"lat": neg_lats, "lon": neg_lons, "date": neg_dates, "label": 0})

dataset = pd.concat([positives, negatives], ignore_index=True)
print(f"Dataset size: {len(dataset)} ({len(positives)} fire, {len(negatives)} no-fire)")

In [ ]:
def fetch_weather_for_point(lat, lon, d):
    """Open-Meteo Historical Weather API — real data, no key needed."""
    resp = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params={
            "latitude": lat, "longitude": lon,
            "start_date": d, "end_date": d,
            "daily": "temperature_2m_max,relative_humidity_2m_mean,wind_speed_10m_max,precipitation_sum",
            "timezone": "Asia/Kolkata",
        },
        timeout=20,
    )
    resp.raise_for_status()
    daily = resp.json().get("daily", {})
    if not daily.get("time"):
        return None
    return {
        "temp": daily["temperature_2m_max"][0],
        "humidity": daily["relative_humidity_2m_mean"][0],
        "wind_speed": daily["wind_speed_10m_max"][0],
        "rainfall": daily["precipitation_sum"][0],
    }

# NOTE: this loop makes one API call per row — for a full run (~thousands of rows)
# expect this cell to take a while. Reduce via dataset.sample(n=...) below for a
# quick first pass.
SAMPLE_SIZE = 500  # reduce API calls for a first test run; raise once verified working
sample = dataset.sample(n=min(SAMPLE_SIZE, len(dataset)), random_state=42).reset_index(drop=True)

weather_rows = []
for i, row in sample.iterrows():
    w = fetch_weather_for_point(row["lat"], row["lon"], row["date"])
    weather_rows.append(w)
    if i % 50 == 0:
        print(f"  {i}/{len(sample)}")
    time.sleep(0.1)

weather_df = pd.DataFrame(weather_rows)
full = pd.concat([sample.reset_index(drop=True), weather_df], axis=1).dropna()
print(f"Final labeled dataset (with weather): {len(full)} rows")
full.head()

## 3. Train XGBoost classifier

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb

FEATURES = ["temp", "humidity", "wind_speed", "rainfall"]
X = full[FEATURES]
y = full["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    eval_metric="logloss", random_state=42,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["No Fire", "Fire"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")

## 4. Feature importance (for thesis/demo credibility)

In [ ]:
import matplotlib.pyplot as plt

importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
importance.plot(kind="barh", title="Feature Importance — Fire Risk Model")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 5. Save the trained model

In [ ]:
import joblib

joblib.dump(model, "fire_risk_model.pkl")
print("Saved fire_risk_model.pkl")

# In Colab: download it to your machine
from google.colab import files
files.download("fire_risk_model.pkl")